In [ ]:
import numpy as np
import jax
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ensure shared widgets/outputs exist (don't overwrite if already defined)
if 'out' not in globals():
    out = widgets.Output()
if 'recycle_slider' not in globals():
    recycle_slider = widgets.IntSlider(value=0, min=0, max=3, description='recycle', continuous_update=False)
if 'loop_slider' not in globals():
    loop_slider = widgets.IntSlider(value=1, min=1, max=48, description='main_loop', continuous_update=False)

def load_pair(recycle, main_loop):
    p1 = f'xcl1_distograms/model_1_recycle_{recycle}_main_loop_{main_loop}_distogram.npz'
    p2 = f'anc0_distograms/model_1_recycle_{recycle}_main_loop_{main_loop}_distogram.npz'
    return np.load(p1), np.load(p2)

def build_xs(bin_edges):
    xs_local = [(2 + bin_edges[0]) / 2]
    for k in range(0, len(bin_edges) - 1):
        xs_local.append((bin_edges[k] + bin_edges[k + 1]) / 2)
    xs_local.append((bin_edges[-1] + 22) / 2)
    return np.array(xs_local)

def update_compare(recycle, main_loop):
    with out:
        clear_output(wait=True)
        d1, d2 = load_pair(recycle, main_loop)

        xs1 = build_xs(d1['bin_edges'])
        xs2 = build_xs(d2['bin_edges'])

        soft1 = jax.nn.softmax(d1['logits'])
        soft2 = jax.nn.softmax(d2['logits'])

        idx1 = np.array(jax.numpy.argmax(soft1, axis=2))
        idx2 = np.array(jax.numpy.argmax(soft2, axis=2))

        dist1 = xs1[idx1]
        dist2 = xs2[idx2]

        vmin = min(dist1.min(), dist2.min())
        vmax = max(dist1.max(), dist2.max())

        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        im0 = axes[0].imshow(dist1, cmap='viridis', interpolation='nearest', vmin=vmin, vmax=vmax)
        axes[0].set_title(f'xcl1: recycle={recycle}, main_loop={main_loop}')
        axes[0].set_xlabel('Residue j')
        axes[0].set_ylabel('Residue i')

        im1 = axes[1].imshow(dist2, cmap='viridis', interpolation='nearest', vmin=vmin, vmax=vmax)
        axes[1].set_title(f'anc0: recycle={recycle}, main_loop={main_loop}')
        axes[1].set_xlabel('Residue j')
        axes[1].set_ylabel('Residue i')

        fig.colorbar(im1, ax=axes.ravel().tolist(), label='Distance (Å)')
        plt.show()

widgets.interact(update_compare, recycle=recycle_slider, main_loop=loop_slider)
display(out)

interactive(children=(IntSlider(value=3, continuous_update=False, description='recycle', max=3), IntSlider(val…

Output()

In [5]:
# two side-by-side line plots: anc0 (left) and xcl1 (right), shared recycle/main_loop, separate i/j sliders
# load a default pair to size the sliders
xcl1_d, anc0_d = load_pair(recycle_slider.value, loop_slider.value)
n_res_anc = anc0_d['logits'].shape[0]
n_res_xcl = xcl1_d['logits'].shape[0]

i_slider_anc = widgets.IntSlider(value=0, min=0, max=n_res_anc-1, description='anc i', continuous_update=False)
j_slider_anc = widgets.IntSlider(value=min(1, n_res_anc-1), min=0, max=n_res_anc-1, description='anc j', continuous_update=False)
i_slider_xcl = widgets.IntSlider(value=0, min=0, max=n_res_xcl-1, description='xcl i', continuous_update=False)
j_slider_xcl = widgets.IntSlider(value=min(1, n_res_xcl-1), min=0, max=n_res_xcl-1, description='xcl j', continuous_update=False)

def update_two_lines(recycle, main_loop, i_anc, j_anc, i_xcl, j_xcl):
    with out_line:
        clear_output(wait=True)
        xcl1_d, anc0_d = load_pair(recycle, main_loop)

        xs_anc = build_xs(anc0_d['bin_edges'])
        xs_xcl = build_xs(xcl1_d['bin_edges'])

        soft_anc = jax.nn.softmax(anc0_d['logits'])
        soft_xcl = jax.nn.softmax(xcl1_d['logits'])

        data_anc = np.array(soft_anc[i_anc, j_anc, :])
        data_xcl = np.array(soft_xcl[i_xcl, j_xcl, :])

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(xs_anc, data_anc, marker='o')
        axes[0].set_title(f'anc0 (i={i_anc}, j={j_anc}) recycle={recycle}, main_loop={main_loop}')
        axes[0].set_xlabel('Bin Centers (Å)')
        axes[0].set_ylabel('Probability')
        axes[0].grid(True)

        axes[1].plot(xs_xcl, data_xcl, marker='o')
        axes[1].set_title(f'xcl1 (i={i_xcl}, j={j_xcl}) recycle={recycle}, main_loop={main_loop}')
        axes[1].set_xlabel('Bin Centers (Å)')
        axes[1].grid(True)

        plt.tight_layout()
        plt.show()

widgets.interact(
    update_two_lines,
    recycle=recycle_slider,
    main_loop=loop_slider,
    i_anc=i_slider_anc,
    j_anc=j_slider_anc,
    i_xcl=i_slider_xcl,
    j_xcl=j_slider_xcl,
)
display(out_line)

interactive(children=(IntSlider(value=3, continuous_update=False, description='recycle', max=3), IntSlider(val…

Output()

In [13]:
import py3Dmol

def view_pdb(pdb_path, style='cartoon', color='spectrum', width=800, height=600):
    """
    Display PDB file at pdb_path in a py3Dmol viewer.
    style: 'cartoon', 'stick', 'sphere', 'line', 'surface', ...
    color: e.g. 'spectrum', 'gray', {'chain':'A'}, etc.
    """
    with open(pdb_path, 'r') as fh:
        pdb = fh.read()
    v = py3Dmol.view(width=width, height=height)
    v.addModel(pdb, 'pdb')
    style_opts = {}
    # cartoon accepts color names like 'spectrum'; sticks/spheres can use color too
    if isinstance(color, dict):
        style_opts.update(color)
    else:
        style_opts['color'] = color
    v.setStyle({}, {style: style_opts})
    v.zoomTo()
    return v.show()

In [37]:
view_pdb('xcl1_out/xcl1_unrelaxed_rank_023_alphafold2_model_5_seed_003.pdb')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.